In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [2]:
unmatched_file = Path(project_root / "data/people/unmatched_flat.json")
new_people_file = Path(project_root / "scripts/notebooks/new_people.json")

In [3]:
with open(unmatched_file, "r") as f:
   unmatched = json.load(f)

with open(new_people_file, "r") as f:
   people = json.load(f)


In [4]:
unmatched_dict = {}
for entry in unmatched:
    composite_id = entry["composite_id"]

    if composite_id not in unmatched_dict:
        unmatched_dict[composite_id] = [entry]
    else:
        unmatched_dict[composite_id].append(entry)

In [5]:
def normalize_name(name):
    name = re.sub(r'\(.*?\)', '', name)   # remove parenthetical content
    name = name.replace(',', '')           # remove commas
    words = sorted(name.strip().split())   # sort words alphabetically
    return ' '.join(words)

unmatched_dict_norm = {}
for composite_id, records in unmatched_dict.items():
    unmatched_dict_norm[composite_id] = []
    for record in records:
        cleaned = record.copy()
        cleaned["display_norm"] = normalize_name(record["display_norm"])
        unmatched_dict_norm[composite_id].append(cleaned)

In [6]:
# rprint(f"total people: {len(people)}")
# rprint(people[:1])
# rprint(dict(list(unmatched_dict.items())[:1]))

# what do I need? get roles from unmatched --> complete info;
# people info: just the basics
# b2p info: flat list with composite_id
people_complete = {}
people_data = []
duplicates = []
duplicate_ids = set()
unexpected_matches = []
no_entries = []
people_count = 0
not_found_count = 0
not_found = {}

for person in people:
# for person in people[:5]:
    name = normalize_name(person["display_norm"])
    composite_ids = person["composite_id"]
    unified_id = person["unified_id"]
    if person["matched_person_id"] or person["match_found"]:
        unexpected_matches.append(person)
        continue
    if unified_id in duplicate_ids:
        duplicates.append(person)
    elif unified_id in people_complete:
        duplicate_ids.add(unified_id)
        duplicates.append(people_complete[unified_id])
        duplicates.append(person)
    else:
        people_count += 1
        people_complete[unified_id] = {
            "unified_id": unified_id,
            "family_name": person["family_name"],
            "given_names": person["given_names"],
            "name_prefix": person["name_prefix"],
            "name_particles": person["name_particles"],
            "name_suffix": person["name_suffix"],
            "single_name": person["single_name"],
            "is_organisation": person["is_organisation"],
            "entries": []
        }

        people_data.append({
            "unified_id": unified_id,
            "family_name": person["family_name"],
            "given_names": person["given_names"],
            "name_prefix": person["name_prefix"],
            "name_particles": person["name_particles"],
            "name_suffix": person["name_suffix"],
            "single_name": person["single_name"],
            "is_organisation": person["is_organisation"],
        })


        for composite_id in composite_ids:
            records = unmatched_dict_norm[composite_id]
            found = False


            for record in records:
                if record["display_norm"] == name:
                    found = True
                    display_name = record["display_name"]
                    sort_order = record["sort_order"]
                    is_author = record["is_author"]
                    is_editor = record["is_editor"]
                    is_contributor = record["is_contributor"]
                    is_translator = record["is_translator"]

                    people_complete[unified_id]["entries"].append({
                        "display_name": display_name,
                        "composite_id": composite_id,
                        "sort_order": sort_order,
                        "is_author": is_author,
                        "is_editor": is_editor,
                        "is_contributor": is_contributor,
                        "is_translator": is_translator,
                    })
                    break

            if not found:
                not_found_count +=1
                found = [r['display_norm'] for r in records]
                not_found[composite_id] = {
                    "searched": name,
                    "found": found
                }
                # rprint(f"no match: {composite_id} | looking for: {name}")
                # rprint(f" display_norm in records is: {[r['display_norm'] for r in records]}")

        people_complete_sorted = dict(sorted(people_complete.items()))
        people_complete = people_complete_sorted

with open("people_complete.json", "w") as f:
    json.dump(people_complete, f, ensure_ascii=False, indent=2)
with open("duplicates.json", "w") as f:
    json.dump(duplicates, f, ensure_ascii=False, indent=2)
with open("not_found.json", "w") as f:
    json.dump(not_found, f, ensure_ascii=False, indent=2)


rprint(f"""
=== LOG ===
people processed: {people_count}
not found: {not_found_count}
duplicates found: {len(duplicates)}
duplicate_ids: {duplicate_ids}
=== DONE ===
""")


=== LOG ===
people processed: 1169
not found: 11
duplicates found: 0
duplicate_ids: set()
=== DONE ===